In [ ]:
#UoB 2023 Lightning
file = "UOB_2023_1.root"
bins = 12

f = ROOT.TFile(file)
t = f.Get("events")
f.Close()

h_2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kYellow-6, bins)

ymax = 1.1 * h_2023.GetMaximum()
h_2023.SetMaximum(ymax)
h_2023.SetMinimum(0)

c_2023 = ROOT.TCanvas("c_2023", "2023", 900, 600)

h_2023.SetTitle("University of Birmingham 2023 Lightning Correlation")
h_2023.Draw("HIST")


lightning_2023 = extract_lightning(Bham_lightning_files["2023"])

year_seconds = seconds_in_year("2023")
month_seconds = year_seconds / 12


lightning_max = max(lightning_2023)
hist_max = h_2023.GetMaximum()

scale = hist_max / lightning_max


g_lightning = ROOT.TGraph(12)

for i, val in enumerate(lightning_2023):

    x = (i + 0.5) * month_seconds
    y = val * scale

    g_lightning.SetPoint(i, x, y)

g_lightning.SetMarkerStyle(20)
g_lightning.SetMarkerSize(1.2)
g_lightning.SetMarkerColor(ROOT.kRed)
g_lightning.SetLineColor(ROOT.kRed)


g_lightning.Draw("P SAME")


axis = ROOT.TGaxis(year_seconds, 0, year_seconds, hist_max, 0, lightning_max, 510, "+L")

axis.SetLineColor(ROOT.kRed)
axis.SetLabelColor(ROOT.kRed)
axis.SetTitle("Lightning hours per month")
axis.Draw()


legend = ROOT.TLegend(0.7,0.75,0.88,0.88)

legend.AddEntry(h_2023,"Events","l")
legend.AddEntry(g_lightning,"Lightning","p")

legend.Draw()


events = np.array([h_2023.GetBinContent(i+1) for i in range(12)])
lightning = lightning_2023

corr = np.corrcoef(events, lightning)[0,1]

print("Lightning vs Event correlation:", corr)


c_2023.Update()
c_2023.Draw()

In [ ]:
def correlation(histograms, lightning_files):
    events_all = []
    lightning_all = []

    for year, hist in histograms:

        events = [hist.GetBinContent(i+1) for i in range(12)]
        lightning = extract_lightning(lightning_files[year])

        events_all.extend(events)
        lightning_all.extend(lightning)

    corr = np.corrcoef(events_all, lightning_all)[0,1]
    return corr

In [ ]:
#Bromsgrove Lightning

offset = 0
histograms = []
bins = 72


for i, year in enumerate(Bromsgrove_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, Bromsgrove_year_files[year], UNIX_year_start, colour, offset, Bromsgrove_total_span, bins)
    histograms.append((year, hist))
    offset += Bromsgrove_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

histograms[0][1].SetTitle("Bromsgrove Lightning Correlation")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")

offset = 0
lightning_graphs = []

for year, h in histograms:
    g, axis = draw_lightning_overlay(year, h, offset, Bham_lightning_files)

    lightning_graphs.append(g)

    offset += Bromsgrove_year_lengths[year]

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

#leg.AddEntry(lightning_graphs[0], "Lightning", "p")

leg.Draw()

c_side.Draw()

corr = correlation(histograms, Bham_lightning_files)
print(f"Correlation (events vs lightning) across all years: {corr:.3f}")